In [7]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import PIL.Image as Image

In [8]:
torch.cuda.is_available()

True

In [19]:
from transformers import AutoFeatureExtractor, ViTForImageClassification
from PIL import Image
from torchvision.transforms import ToTensor
import requests
import os

image = Image.open("../data/Frame_25.png")
feature_extractor = AutoFeatureExtractor.from_pretrained('facebook/deit-tiny-patch16-224')
# model = ViTForImageClassification.from_pretrained('facebook/deit-tiny-patch16-224')
inputs = feature_extractor(images=np.array(image)[:, :, :3], return_tensors="pt")
# outputs = model(**inputs)
# logits = outputs.logits
# model predicts one of the 1000 ImageNet classes
# predicted_class_idx = logits.argmax(-1).item()
# print("Predicted class:", model.config.id2label[predicted_class_idx])
print(inputs)

{'pixel_values': tensor([[[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]],

         [[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]],

         [[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]]]])}


In [7]:
from transformers import CLIPModel, CLIPProcessor
_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
img = Image.open('../data/Frame_25.png').convert('RGB')
inputs = _processor(images=img, return_tensors='pt', padding=True)
with torch.no_grad():
    vision_outputs = _model.vision_model(**inputs)
    image_embeds = vision_outputs[1]
    image_embeds = _model.visual_projection(image_embeds)
    image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True) 
print(image_embeds.shape)

torch.Size([1, 512])


In [9]:
import os, sys
sys.path.append(os.path.abspath("/home/ranai/MRSD/diffusIn/include"))
from torchvision.transforms.v2 import Compose, Resize, ToTensor, Normalize
from torchvision.transforms.functional import adjust_brightness

import torch
from OpenVision.src.convert_upload.open_clip.factory import create_vision_encoder_and_transforms

# Replace with your uploaded repo name
hf_repo = "UCSC-VLAA/openvision-vit-tiny-patch16-384"

# Load converted vision encoder
vision_encoder = create_vision_encoder_and_transforms(
    model_name=f"hf-hub:{hf_repo}"
)

# Run inference
vision_encoder.eval()
img = Image.open('../data/Frame_30.png').convert('RGB')
tensor_conv = Compose([
    Resize((384, 384)),
    ToTensor(),
    Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
              std=[0.26862954, 0.26130258, 0.27577711]),
])

with torch.no_grad():
    patch_features = vision_encoder(torch.unsqueeze(adjust_brightness(tensor_conv(img),1.05), 0))

print("Patch feature shape:", patch_features[0, :].shape)

Patch feature shape: torch.Size([192])


/home/ranai/miniconda3/envs/idl/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [5]:
import h5py

path = "../data/episode_0.hdf5"

with h5py.File(path, 'r') as root:
    print("Attributes:")
    for key, value in root.attrs.items():
        print(f"  {key}: {value}")

    print(root.keys)

Attributes:
  sim: True
<bound method MappingHDF5.keys of <HDF5 file "episode_0.hdf5" (mode r)>>
